In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [10]:
sns.set_theme(style="darkgrid")

In [5]:
df_master=pd.read_csv('C:/CLV-Project/data/cleaned_data_/master_feature_matrix.csv')
df_master.head()

,id,username,num_bookings,total_ticket_spend,avg_ticket_spend,num_tickets_bought,fav_theater,fav_showtime,num_snack_orders,total_snack_spend,avg_snack_spend,num_reviews,avg_rating,num_chat_sessions,fav_theater_enc,fav_showtime_enc,clv_score,clv_normalized
0,7,marcus_aurelius,9,435.0,48.33,30,Apollo Theater,8:00 PM,8.0,88.0,11.00,2.0,5.0,3,0,3,548.0,100.00
1,3,tony_stark,9,420.5,46.72,29,Apollo Theater,2:30 PM,7.0,99.5,14.21,2.0,5.0,3,0,1,545.0,99.42
2,6,alexandra_daddario,9,391.5,43.50,27,Apollo Theater,10:45 PM,9.0,120.0,13.33,2.0,4.5,3,0,0,534.0,97.29
3,2,sarah_connor,9,420.5,46.72,29,IMAX Nebula,5:15 PM,7.0,88.5,12.64,2.0,4.5,3,2,2,531.5,96.80
4,10,sophia_chen,9,420.5,46.72,29,Apollo Theater,10:45 PM,7.0,81.5,11.64,2.0,4.0,3,0,0,522.0,94.96


### Defining Features(X) and Target (y)

In [6]:
FEATURES = [
    'num_bookings', 
    'total_ticket_spend', 
    'avg_ticket_spend', 
    'num_tickets_bought', 
    'num_snack_orders', 
    'total_snack_spend', 
    'avg_snack_spend', 
    'num_reviews', 
    'avg_rating', 
    'num_chat_sessions', 
    'fav_theater_enc', 
    'fav_showtime_enc'
]
X = df_master[FEATURES]
y = df_master['clv_score']

In [8]:
# 75% Training / 25% Testing Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [9]:
print(f"Dataset Split: Training set = {len(X_train)} samples | Testing set = {len(X_test)} samples")


Dataset Split: Training set = 33 samples | Testing set = 12 samples


### Benchemarking Ensemble Models

In [12]:
candidate_models = {
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Extra Trees':       ExtraTreesRegressor(n_estimators=100, random_state=42)
}
benchmark_results = []
best_r2_score = -1.0
best_model_name = ""
best_model_obj = None
for name, model in candidate_models.items():
    # Fit model
    model.fit(X_train, y_train)
    
    # Predict on test set
    predictions = model.predict(X_test)
    
    # Metrics
    mae  = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2   = r2_score(y_test, predictions)
    
    benchmark_results.append({
        'Model Algorithm': name,
        'MAE ($)': round(mae, 2),
        'RMSE ($)': round(rmse, 2),
        'R² Score (Accuracy)': round(r2, 4)
    })
    
    print(f"  ├─ {name:18s} | MAE: ${mae:6.2f} | RMSE: ${rmse:6.2f} | R²: {r2:.4f}")
    
    if r2 > best_r2_score:
        best_r2_score = r2
        best_model_name = name
        best_model_obj = model
results_df = pd.DataFrame(benchmark_results).sort_values('R² Score (Accuracy)', ascending=False)
print("\n" + "=" * 60)
print("🏆 MODEL BENCHMARK PERFORMANCE COMPARISON")
print("=" * 60)
print(results_df.to_string(index=False))

  ├─ Random Forest      | MAE: $  6.80 | RMSE: $  9.21 | R²: 0.9978
  ├─ Gradient Boosting  | MAE: $  9.22 | RMSE: $ 12.03 | R²: 0.9962
  ├─ Extra Trees        | MAE: $  6.69 | RMSE: $  9.29 | R²: 0.9977

🏆 MODEL BENCHMARK PERFORMANCE COMPARISON
  Model Algorithm  MAE ($)  RMSE ($)  R² Score (Accuracy)
    Random Forest     6.80      9.21               0.9978
      Extra Trees     6.69      9.29               0.9977
Gradient Boosting     9.22     12.03               0.9962


### Serealizing Best Model

In [14]:
MODEL_PATH = 'C:/CLV-Project/data/models/clv_best_model.pkl'
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(best_model_obj, f)
print(f"\n💾 Winning Model ({best_model_name}) serialized & saved to {MODEL_PATH}")


💾 Winning Model (Random Forest) serialized & saved to C:/CLV-Project/data/models/clv_best_model.pkl


### Feature Importance Analysis

In [15]:
importances = pd.Series(best_model_obj.feature_importances_, index=FEATURES).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
bars = plt.barh(importances.index, importances.values * 100, color='teal', edgecolor='black')
plt.title(f'Feature Importance Breakdown ({best_model_name})', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Relative Importance Weight (%)', fontsize=11, fontweight='bold')
plt.ylabel('Customer Feature', fontsize=11, fontweight='bold')
# Annotate percentage values on bars
for bar in bars:
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
             f'{bar.get_width():.1f}%', va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
chart_path = 'C:/CLV-Project/data/charts/07_feature_importance.png'
plt.savefig(chart_path, dpi=300)
plt.close()